# DataFrames 2: Polars expressions and lazy queries

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/dataframes/dataframes-02-polars.ipynb)

Filter orders, aggregate revenue, and inspect the plan that reads the input.

*Last reviewed: November 2026, against Polars 2.1.*

In [ ]:
# Setup: install notebook dependencies in Colab.
!pip install -q polars~=2.1.0

## Setup and sample data

Four order rows include a draft with a missing amount. Save them as Parquet so the lazy reader can optimize the file scan.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import polars as pl

workspace = TemporaryDirectory()
orders_path = Path(workspace.name) / "orders.parquet"
orders = pl.DataFrame({
    "id": [1, 2, 3, 4],
    "region": ["west", "east", "west", "east"],
    "status": ["paid", "paid", "draft", "paid"],
    "cents": [1000, 1500, None, 2000],
})
orders.write_parquet(orders_path)
print(orders)

## Build a column expression

An expression describes how to compute or filter a column. Pass it to a DataFrame method to apply it to all rows.

**Where it matters**

Filter valid paid orders before aggregating revenue.

**Watch out for**

Decide what missing money means before filling or aggregating it.

In [ ]:
paid = orders.filter((pl.col("status") == "paid") & pl.col("cents").is_not_null())
print(paid.group_by("region").agg(revenue_cents=pl.col("cents").sum()).sort("region"))

**What to notice:** East totals 3500 cents and west 1000 cents. The draft contributes no revenue.

## Scan lazily and inspect the plan

scan_parquet builds a lazy query. explain shows the plan and collect executes it. Projection and predicate pushdown can reduce scan work.

**Where it matters**

Inspect which columns and filters reach a Parquet scan.

**Watch out for**

Starting with read_parquet loads the file eagerly before later operations can be pushed into the scan.

In [ ]:
query = (pl.scan_parquet(orders_path)
         .filter(pl.col("status") == "paid")
         .select("id", "cents"))
print(query.explain())
print(query.collect())

**What to notice:** The scan projects three columns, including status for filtering. The final output has only id and cents.

## Check yourself

### Exercise 1: Apply the pattern

**Difficulty:** Easy · about 5 min

Count paid rows by region. Expect east 2 and west 1.

In [ ]:
# Try your solution here.

In [ ]:
print(paid.group_by("region").len().sort("region"))

## Key takeaways

- Filter valid paid orders before aggregating revenue.
- Inspect which columns and filters reach a Parquet scan.

**Next:** [DataFrames 3: Arrow](dataframes-03-arrow.ipynb)